# Step 24. Paired group bootstrap between models

Paired group bootstrap of the difference between two models on the same test rows (the plan's "Đánh giá chênh lệch
theo cặp"). Part 1 pairs models from experiment A inside the same (dataset, cohort, protocol, split seed, model seed).
Part 2 loads the legacy frozen-benchmark predictions (hybrid, No-G, MLP, LightGBM, E-GraphSAGE) at the fixed split
seed 2027, maps them to dataset rows through `split_indices.npz` of step 5 and the subset definition, and accepts a
file only when its labels and family names match the test rows in order. Edit the `LEGACY_MODELS` paths to the Drive
folders of steps 6 and 7 before running Part 2.

Each draw resamples test groups with replacement and applies the same multiplicities to both models; the 2.5 % and
97.5 % percentiles of the metric difference are reported per model seed, with the observed delta. Row-weighted and
group-balanced versions are both computed. These intervals are not confidence intervals for the mean over seeds.

In [1]:
!pip -q install lightgbm==4.6.0
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
import numpy, pandas, lightgbm
print('pandas', pandas.__version__, '| numpy', numpy.__version__, '| lightgbm', lightgbm.__version__)
try:
    import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
except ImportError:
    print('torch not installed: the MLP runs need it')

Mounted at /content/drive
pandas 2.2.3 | numpy 2.4.6 | lightgbm 4.6.0
torch 2.14.1+cpu | cuda False


## Configuration

In [2]:
# Paired group bootstrap of the DIFFERENCE between two models scored on the same test rows.
# Part 1: pairs of models from experiment A (same dataset, cohort, protocol, split_seed, model_seed).
# Part 2: legacy frozen-benchmark predictions (hybrid, LightGBM, E-GraphSAGE, ...) at the fixed split
#         seed 2027, aligned to the subset test rows through split_indices.npz and checked on labels and families.
DATA_DIR    = '/content/drive/MyDrive'
RUN_ROOT    = '/content/drive/MyDrive/THAT_IDS_RERUN'
SUBSET_ROOT = f'{RUN_ROOT}/conference_step5_literature_subset_v1/subset_definition'
STEP_A_DIR  = f'{RUN_ROOT}/journal_step22_rr_vs_gd_v1'
OUT_DIR     = f'{RUN_ROOT}/journal_step24_paired_model_bootstrap_v1'
DATASETS_TO_RUN = ['UNSW_V2', 'CIC_V2']
N_BOOT, BOOT_SEED = 2000, 20261111
GROUP_BALANCED_TOO = True

# Part 1: model pairs inside experiment A
A_PAIRS = [('lightgbm', 'mlp')]

# Part 2: legacy prediction files. {ds} is the dataset name, {seed} the model seed. Edit the roots to match Drive.
LEGACY_SPLIT = f'{RUN_ROOT}/conference_step5_literature_subset_v1/confirmation/{{ds}}/exact_feature_group/shared/seed_2027/split_indices.npz'
LEGACY_MODELS = {
    'hybrid':     f'{RUN_ROOT}/conference_step5_literature_subset_v1/confirmation/{{ds}}/exact_feature_group/full/seed_{{seed}}/test_predictions.npz',
    'no_graph':   f'{RUN_ROOT}/conference_step5_literature_subset_v1/confirmation/{{ds}}/exact_feature_group/no_graph/seed_{{seed}}/test_predictions.npz',
    'mlp':        f'{RUN_ROOT}/conference_step5_literature_subset_v1/confirmation/{{ds}}/exact_feature_group/mlp/seed_{{seed}}/test_predictions.npz',
    'lightgbm':   f'{RUN_ROOT}/conference_step7_tabular_loo_v1/{{ds}}/exact_feature_group/lightgbm/seed_{{seed}}/test_predictions.npz',
    'egraphsage': f'{RUN_ROOT}/conference_step6_public_baselines_v1/{{ds}}/exact_feature_group/egraphsage_public/seed_{{seed}}/test_predictions.npz',
}
LEGACY_PAIRS = [('hybrid', 'lightgbm'), ('hybrid', 'egraphsage'), ('hybrid', 'no_graph'), ('hybrid', 'mlp'), ('lightgbm', 'egraphsage')]
LEGACY_MODEL_SEEDS = [42, 123, 456]
os.makedirs(os.path.join(OUT_DIR, 'reports'), exist_ok=True)

## Shared code

In [3]:
# ---- Shared code: data loading, hash groups, splits, thresholds, metrics, bootstrap ----
# Copied verbatim into each experiment notebook so that every notebook is self-contained.
import os, json, time, gc, math, hashlib, warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score

EXCLUDE_COLS = ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR',
                'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT', 'Label', 'Attack']
N_FEATURES = 39
CHUNK = 1_000_000
TRAIN_FRAC, VAL_FRAC = 0.70, 0.10
TARGET_SPLIT_FRACTIONS = np.asarray([0.70, 0.10, 0.20], dtype=np.float64)
N_CANDIDATE_SALTS = 24
MIN_FAMILY_TOTAL_FOR_BALANCE = 30
SHUFFLE_SEED_OFFSET = 17_171
SPLIT_NAMES = ('train', 'validation', 'test')

DATASETS = {
    'UNSW_V2': dict(file='NF-UNSW-NB15-v2.csv', rows=2_390_275, groups=236_999,
                    frozen_rows=245_994, frozen_groups=41_640),
    'CIC_V2':  dict(file='NF-CSE-CIC-IDS2018-v2.csv', rows=18_893_708, groups=8_981_062,
                    frozen_rows=496_384, frozen_groups=233_966),
}
# Salts and (rows, benign, attack) of the six stored full-data group-disjoint splits
# (splits/paper_full_data_splits.json in the published package).
PAPER_FULL_SPLITS = {
    ('UNSW_V2', 42):  dict(salt=11652151511078254891,
        train=(1670959, 1604425, 66534), validation=(248331, 235791, 12540), test=(470985, 455006, 15979)),
    ('UNSW_V2', 123): dict(salt=4518799303034446728,
        train=(1669960, 1600374, 69586), validation=(219285, 210798, 8487), test=(501030, 484050, 16980)),
    ('UNSW_V2', 456): dict(salt=5346391484498864780,
        train=(1667966, 1600280, 67686), validation=(269673, 260319, 9354), test=(452636, 434623, 18013)),
    ('CIC_V2', 42):   dict(salt=8095878257575067585,
        train=(13232623, 11630102, 1602521), validation=(2012758, 1796541, 216217), test=(3648327, 3208924, 439403)),
    ('CIC_V2', 123):  dict(salt=9558457822766297306,
        train=(13254901, 11705849, 1549052), validation=(1893721, 1659711, 234010), test=(3745086, 3270007, 475079)),
    ('CIC_V2', 456):  dict(salt=3253095538486194220,
        train=(13336590, 11700738, 1635852), validation=(1836131, 1621337, 214794), test=(3720987, 3313492, 407495)),
}


def log(msg):
    print(time.strftime('%H:%M:%S'), msg, flush=True)


def atomic_json(obj, path):
    tmp = str(path) + '.tmp'
    with open(tmp, 'w') as fh:
        json.dump(obj, fh, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    os.replace(tmp, path)


# ---------------------------------------------------------------- data ----
def load_dataset(name, data_dir):
    """Chunked reader. Returns X (float32, 39 cols), uint64 group keys, binary labels,
    family codes, family names and the replaced-value count per column."""
    cfg = DATASETS[name]
    path = os.path.join(data_dir, cfg['file'])
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path}: edit DATA_DIR in the config cell')
    header = pd.read_csv(path, nrows=0).columns.tolist()
    feat_cols = [c for c in header if c not in EXCLUDE_COLS]
    assert len(feat_cols) == N_FEATURES, f'{len(feat_cols)} feature columns, expected {N_FEATURES}'
    n = cfg['rows']
    X = np.empty((n, N_FEATURES), dtype=np.float32)
    keys = np.empty(n, dtype=np.uint64)
    y = np.empty(n, dtype=np.int8)
    fam = np.empty(n, dtype=np.int16)
    replaced = np.zeros(N_FEATURES, dtype=np.int64)
    fam_names, pos, t0 = {}, 0, time.time()
    for chunk in pd.read_csv(path, usecols=feat_cols + ['Label', 'Attack'], chunksize=CHUNK):
        f = chunk[feat_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
        a = f.to_numpy(copy=True)
        bad = ~np.isfinite(a)
        replaced += bad.sum(axis=0)
        a[bad] = 0.0
        f = pd.DataFrame(a, columns=feat_cols)
        m = len(f)
        if pos + m > n:
            raise ValueError(f'{name}: more rows than expected ({n})')
        X[pos:pos + m] = a
        keys[pos:pos + m] = pd.util.hash_pandas_object(f, index=False).to_numpy()
        y[pos:pos + m] = chunk['Label'].to_numpy().astype(np.int8)
        att = chunk['Attack'].astype(str)
        for v in att.unique():
            fam_names.setdefault(v, len(fam_names))
        fam[pos:pos + m] = att.map(fam_names).to_numpy().astype(np.int16)
        pos += m
        print(f'\r{name}: {pos:,} rows read ({time.time() - t0:.0f}s)', end='')
    print()
    assert pos == n, f'{name}: read {pos:,} rows, expected {n:,}'
    n_groups = len(np.unique(keys))
    log(f'{name}: groups {n_groups:,} (audit {cfg["groups"]:,}) | repeated-row rate '
        f'{100 * (1 - n_groups / n):.2f}% | attack {100 * y.mean():.2f}% | replaced values {int(replaced.sum()):,}')
    if n_groups != cfg['groups']:
        print('  WARNING: group count differs from the audit; preprocessing does not match the paper.')
    names = [k for k, _ in sorted(fam_names.items(), key=lambda kv: kv[1])]
    return dict(name=name, X=X, keys=keys, y=y, fam=fam, names=np.array(names), feat_cols=feat_cols,
                n_groups=n_groups, replaced_per_column=dict(zip(feat_cols, replaced.tolist())))


def load_frozen_cohort(name, data, subset_root):
    """Row indices of the frozen attack-enriched subset (subset_indices_int32.npy of step 5)."""
    cfg = DATASETS[name]
    path = os.path.join(subset_root, name, 'subset_indices_int32.npy')
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path}: the frozen subset definition is needed for the frozen cohort')
    idx = np.load(path).astype(np.int64)
    assert len(idx) == cfg['frozen_rows'], f'{name}: subset has {len(idx):,} rows, expected {cfg["frozen_rows"]:,}'
    assert len(np.unique(idx)) == len(idx), 'duplicate row indices in the subset'
    sub_keys = data['keys'][idx]
    n_groups = len(np.unique(sub_keys))
    assert n_groups == cfg['frozen_groups'], f'{name}: subset has {n_groups:,} groups, expected {cfg["frozen_groups"]:,}'
    # complete-group check: every group of the subset must be entirely inside it
    full_counts = pd.Series(data['keys']).value_counts()
    sub_counts = pd.Series(sub_keys).value_counts()
    assert (full_counts.loc[sub_counts.index].to_numpy() == sub_counts.to_numpy()).all(), 'subset contains partial groups'
    log(f'{name}: frozen cohort {len(idx):,} rows, {n_groups:,} complete groups, attack {100 * data["y"][idx].mean():.2f}%')
    return idx


# --------------------------------------------------------------- splits ----
def splitmix64(values):
    x = np.asarray(values, dtype=np.uint64).copy()
    with np.errstate(over='ignore'):
        x = x + np.uint64(0x9E3779B97F4A7C15)
        x = (x ^ (x >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
        x = (x ^ (x >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
        x = x ^ (x >> np.uint64(31))
    return x


def assign_groups(unique_hashes, salt):
    mixed = splitmix64(np.asarray(unique_hashes) ^ np.uint64(salt))
    unit = mixed.astype(np.float64) / float(2 ** 64)
    return np.where(unit < TRAIN_FRAC, 0, np.where(unit < TRAIN_FRAC + VAL_FRAC, 1, 2)).astype(np.int8)


def assignment_objective(group_assignment, group_inverse, group_counts, family_codes, n_families):
    counts = np.asarray(group_counts, dtype=np.int64)
    split_rows = np.bincount(group_assignment.astype(np.int64), weights=counts, minlength=3).astype(np.int64)
    split_code = group_assignment[np.asarray(group_inverse)].astype(np.int64)
    combined = split_code * n_families + np.asarray(family_codes).astype(np.int64)
    split_family = np.bincount(combined, minlength=3 * n_families).reshape(3, n_families)
    row_fractions = split_rows / max(split_rows.sum(), 1)
    global_error = float(np.abs(row_fractions - TARGET_SPLIT_FRACTIONS).mean())
    totals = split_family.sum(axis=0)
    eligible = totals >= MIN_FAMILY_TOTAL_FOR_BALANCE
    family_fraction = np.divide(split_family, totals[None, :], out=np.zeros_like(split_family, dtype=np.float64),
                                where=totals[None, :] > 0)
    family_error = np.abs(family_fraction - TARGET_SPLIT_FRACTIONS[:, None])
    if eligible.any():
        weights = np.sqrt(totals[eligible].astype(np.float64)); weights /= weights.sum()
        family_mae = float((family_error[:, eligible].mean(axis=0) * weights).sum())
        max_family_error = float(family_error[:, eligible].max())
    else:
        family_mae = max_family_error = 0.0
    expected = TARGET_SPLIT_FRACTIONS[:, None] * totals[None, :]
    missing_required = int(((split_family == 0) & (expected >= 5)).sum())
    objective = 5.0 * global_error + family_mae + 0.25 * max_family_error + 10.0 * missing_required
    return dict(objective=float(objective), global_fraction_mae=global_error, family_fraction_weighted_mae=family_mae,
                max_family_fraction_error=max_family_error, missing_required_family_cells=missing_required,
                split_rows=split_rows.tolist())


def hash_group_split(keys, family_codes, seed, n_candidates=N_CANDIDATE_SALTS, salt=None):
    """Algorithm 1. Returns part (0 train / 1 val / 2 test per row, in input order), salt, metadata.
    Rows inside each split are shuffled with default_rng(seed + 17171) when order is requested."""
    unique_hashes, inverse, counts = np.unique(np.asarray(keys), return_inverse=True, return_counts=True)
    n_families = int(np.asarray(family_codes).max()) + 1
    salts = (np.random.default_rng(seed).integers(1, np.iinfo(np.uint64).max, size=n_candidates, dtype=np.uint64)
             if salt is None else np.asarray([salt], dtype=np.uint64))
    best = best_assignment = best_salt = None
    candidates = []
    for i, cand in enumerate(salts):
        assignment = assign_groups(unique_hashes, cand)
        m = assignment_objective(assignment, inverse, counts, family_codes, n_families)
        candidates.append(dict(candidate=i, salt=str(int(cand)), **m))
        if best is None or m['objective'] < best['objective']:   # first minimum wins
            best, best_assignment, best_salt = m, assignment, int(cand)
    part = best_assignment[inverse]
    meta = dict(seed=int(seed), n_groups=int(len(unique_hashes)), candidate_salts_tested=int(len(salts)),
                selected_salt=str(best_salt), **{f'selected_{k}': v for k, v in best.items()})
    return part, best_salt, meta, candidates


def split_order(part, seed):
    """Row indices of train/val/test, shuffled inside each split as in notebook 01."""
    rng = np.random.default_rng(seed + SHUFFLE_SEED_OFFSET)
    return tuple(rng.permutation(np.flatnonzero(part == i)) for i in range(3))


def random_row_split(family_codes, seed):
    """Random-row baseline of notebook 01: stratified by family, 70/10/20, one generator per seed."""
    codes = np.asarray(family_codes)
    order = np.argsort(codes, kind='stable')
    counts = np.bincount(codes.astype(np.int64))
    offsets = np.r_[0, np.cumsum(counts)]
    rng = np.random.default_rng(seed)
    tr, va, te = [], [], []
    for i in range(len(counts)):
        perm = rng.permutation(order[offsets[i]:offsets[i + 1]].astype(np.int64))
        c1, c2 = int(len(perm) * TRAIN_FRAC), int(len(perm) * (TRAIN_FRAC + VAL_FRAC))
        tr.append(perm[:c1]); va.append(perm[c1:c2]); te.append(perm[c2:])
    part = np.empty(len(codes), dtype=np.int8)
    part[np.concatenate(tr)] = 0; part[np.concatenate(va)] = 1; part[np.concatenate(te)] = 2
    return part


def split_audit(keys, y, fam, names, part):
    """Rows, groups, attack rate, overlap in both directions, absent families, largest test group."""
    out = {}
    uniq = [np.unique(keys[part == i]) for i in range(3)]
    for i, s in enumerate(SPLIT_NAMES):
        m = part == i
        out[f'{s}_rows'] = int(m.sum()); out[f'{s}_groups'] = int(len(uniq[i]))
        out[f'{s}_attack_pct'] = float(100 * y[m].mean()) if m.any() else float('nan')
        out[f'{s}_families_absent'] = ';'.join(names[f] for f in range(len(names)) if not (fam[m] == f).any())
    for (a, b) in [(0, 2), (0, 1), (1, 2)]:
        src, dst = SPLIT_NAMES[a], SPLIT_NAMES[b]
        m = part == b
        seen = np.isin(keys[m], uniq[a])
        out[f'{src}_to_{dst}_row_overlap_pct'] = float(100 * seen.mean()) if m.any() else float('nan')
        out[f'{src}_to_{dst}_group_overlap_pct'] = float(100 * np.isin(uniq[b], uniq[a]).mean()) if len(uniq[b]) else float('nan')
    te = part == 2
    if te.any():
        _, c = np.unique(keys[te], return_counts=True)
        out['test_largest_group_rows'] = int(c.max())
    return out


def split_fingerprint(part, row_index=None):
    h = hashlib.sha256(part.astype(np.int8).tobytes())
    if row_index is not None:
        h.update(np.asarray(row_index, dtype=np.int64).tobytes())
    return h.hexdigest()[:20]


# ----------------------------------------------------------- thresholds ----
def _curve(y, s):
    order = np.argsort(-s, kind='stable')
    ys, ss = y[order].astype(np.int64), s[order]
    tp, fp = np.cumsum(ys), np.cumsum(1 - ys)
    last = np.r_[ss[1:] != ss[:-1], True]       # last row of each distinct score block
    tp, fp, thr = tp[last], fp[last], ss[last]
    P = int(ys.sum()); N = len(ys) - P
    return tp, fp, thr, P, N


def best_f1_macro_threshold(y, s):
    """Prediction is score >= threshold. Exact search over distinct validation scores plus the
    all-benign rule (threshold = +inf). Ties are broken toward the largest threshold."""
    tp, fp, thr, P, N = _curve(y, s)
    fn, tn = P - tp, N - fp
    with np.errstate(invalid='ignore', divide='ignore'):
        f1 = 0.5 * (np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
                    + np.where(2 * tn + fn + fp > 0, 2 * tn / np.maximum(2 * tn + fn + fp, 1), 0.0))
    f1_none = 0.5 * (0.0 + (2 * N / (2 * N + P) if N + P else 0.0))     # predict all benign
    best = int(np.argmax(f1))                                          # first max = largest threshold
    if f1_none > f1[best]:
        return float('inf'), float(f1_none)
    return float(thr[best]), float(f1[best])


def threshold_at_fpr(y, s, target_fpr):
    """Largest-TPR threshold among those with validation FPR <= target (fraction); ties -> largest
    threshold; +inf if no threshold qualifies. Returns threshold and achieved validation FPR/TPR."""
    tp, fp, thr, P, N = _curve(y, s)
    fpr = fp / max(N, 1); tpr = tp / max(P, 1)
    ok = fpr <= target_fpr
    if not ok.any():
        return float('inf'), 0.0, 0.0
    best = int(np.flatnonzero(ok)[np.argmax(tpr[ok])])                 # first max = largest thr
    return float(thr[best]), float(fpr[best]), float(tpr[best])


# -------------------------------------------------------------- metrics ----
def metrics(y, s, pred, w=None):
    """Counts and rates; w = per-row weights (None = each row counts 1). Undefined ratios are NaN."""
    y = y.astype(np.int64); pred = pred.astype(np.int64)
    w = np.ones(len(y)) if w is None else np.asarray(w, dtype=np.float64)
    tp = float(w[(pred == 1) & (y == 1)].sum()); fp = float(w[(pred == 1) & (y == 0)].sum())
    fn = float(w[(pred == 0) & (y == 1)].sum()); tn = float(w[(pred == 0) & (y == 0)].sum())
    div = lambda a, b: a / b if b > 0 else float('nan')
    f1_att = div(2 * tp, 2 * tp + fp + fn); f1_ben = div(2 * tn, 2 * tn + fn + fp)
    out = dict(n=float(w.sum()), n_attack=tp + fn, n_benign=fp + tn, attack_pct=100 * div(tp + fn, w.sum()),
               f1_macro=0.5 * (f1_att + f1_ben), f1_attack=f1_att, fpr_pct=100 * div(fp, fp + tn),
               tpr=div(tp, tp + fn), precision=div(tp, tp + fp), tp=tp, fp=fp, fn=fn, tn=tn)
    out['pr_auc'] = (float(average_precision_score(y, s, sample_weight=w)) if s is not None and 0 < tp + fn < w.sum()
                     else float('nan'))
    return out


def group_balanced_weights(keys_eval):
    """Weight 1 / (group size inside the evaluation set): every group sums to 1."""
    _, inv, cnt = np.unique(keys_eval, return_inverse=True, return_counts=True)
    return 1.0 / cnt[inv]


def family_recall(y, pred, fam, names, min_rows=1):
    rows = []
    for f in range(len(names)):
        m = (fam == f) & (y == 1)
        n = int(m.sum())
        rows.append(dict(family=str(names[f]), count=n, recall=(float(pred[m].mean()) if n >= min_rows else float('nan'))))
    return rows


def evaluate_run(y_te, s_te, keys_te, fam_te, names, y_va, s_va, fpr_targets=(0.001, 0.0001)):
    """Everything derived from saved scores: F1-macro threshold, FPR-target thresholds,
    row-weighted and group-balanced metrics, per-family recall."""
    thr, f1_val = best_f1_macro_threshold(y_va, s_va)
    pred = s_te >= thr
    out = dict(threshold=thr, val_f1_macro_at_threshold=f1_val)
    out.update(metrics(y_te, s_te, pred))
    gw = group_balanced_weights(keys_te)
    out.update({f'gb_{k}': v for k, v in metrics(y_te, s_te, pred, gw).items()
                if k in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr', 'precision')})
    for t in fpr_targets:
        tag = f'fpr{t * 100:g}'
        n_ben_val = int((y_va == 0).sum())
        th, vf, vt = threshold_at_fpr(y_va, s_va, t)
        p = s_te >= th
        m = metrics(y_te, None, p)
        out.update({f'{tag}_threshold': th, f'{tag}_val_fpr_pct': 100 * vf, f'{tag}_val_tpr': vt,
                    f'{tag}_val_benign_rows': n_ben_val, f'{tag}_test_fpr_pct': m['fpr_pct'],
                    f'{tag}_test_fp': m['fp'], f'{tag}_test_tpr': m['tpr'],
                    f'{tag}_interpretable': bool(n_ben_val * t >= 10)})   # at least 10 expected FP on validation
    out['family_recall'] = family_recall(y_te, pred, fam_te, names)
    return out, pred


def minmax_fit(Xtr):
    mn, mx = Xtr.min(0), Xtr.max(0)
    rng_ = np.where(mx > mn, mx - mn, 1.0).astype(np.float32)
    return mn, rng_


def minmax_apply(X, mn, rng_, clip=None):
    Z = (X - mn) / rng_
    if clip is not None:
        np.clip(Z, clip[0], clip[1], out=Z)
    return Z


# ------------------------------------------------------------ bootstrap ----
def paired_group_bootstrap(keys, y, scores, n_boot=2000, seed=20261111, group_balanced=False):
    """Paired bootstrap over groups of one fixed evaluation set.
    scores: dict condition -> (score array, threshold). The same group multiplicities are
    applied to every condition in each draw. Returns per-draw metrics (dict of arrays) and the number
    of draws in which a class was missing (those draws are left as NaN)."""
    uniq, inv, cnt = np.unique(keys, return_inverse=True, return_counts=True)
    G = len(uniq)
    rng = np.random.default_rng(seed)
    base_w = (1.0 / cnt[inv]) if group_balanced else np.ones(len(keys))
    conds = list(scores)
    preds = {c: (scores[c][0] >= scores[c][1]) for c in conds}
    y64 = y.astype(np.int64)
    store = {c: {k: np.full(n_boot, np.nan) for k in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr')} for c in conds}
    invalid = 0
    for b in range(n_boot):
        mult = np.bincount(rng.integers(0, G, G), minlength=G)
        w = base_w * mult[inv]
        if w[y64 == 1].sum() == 0 or w[y64 == 0].sum() == 0:
            invalid += 1
            continue
        for c in conds:
            m = metrics(y64, scores[c][0], preds[c], w)
            for k in store[c]:
                store[c][k][b] = m[k]
    return store, invalid


def bootstrap_delta_table(store, pairs, invalid, n_boot):
    rows = []
    for a, b in pairs:
        for k in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr'):
            d = store[a][k] - store[b][k]
            d = d[~np.isnan(d)]
            rows.append(dict(comparison=f'{a} - {b}', metric=k, n_valid=int(len(d)), n_invalid=int(invalid),
                             delta_boot_mean=float(d.mean()) if len(d) else float('nan'),
                             ci95_low=float(np.percentile(d, 2.5)) if len(d) else float('nan'),
                             ci95_high=float(np.percentile(d, 97.5)) if len(d) else float('nan')))
    return pd.DataFrame(rows)


## Part 1: experiment A pairs

In [4]:
def load_pred(path):
    z = np.load(path, allow_pickle=True)
    return z


rows_out = []
pred_dir = os.path.join(STEP_A_DIR, 'predictions')
if os.path.isdir(pred_dir):
    files = sorted(f for f in os.listdir(pred_dir) if f.endswith('.npz'))
    index = {}
    for f in files:
        ds, cohort, protocol, ss, model, ms = f[:-4].split('__')
        index.setdefault((ds, cohort, protocol, ss, ms), {})[model] = os.path.join(pred_dir, f)
    for key, models in sorted(index.items()):
        for a, b in A_PAIRS:
            if a in models and b in models:
                za, zb = load_pred(models[a]), load_pred(models[b])
                assert np.array_equal(za['row_index'], zb['row_index']) and np.array_equal(za['y_true'], zb['y_true'])
                scores = {a: (za['score'], float(za['threshold'])), b: (zb['score'], float(zb['threshold']))}
                for gb in ([False, True] if GROUP_BALANCED_TOO else [False]):
                    store, invalid = paired_group_bootstrap(za['key'], za['y_true'], scores, n_boot=N_BOOT, seed=BOOT_SEED, group_balanced=gb)
                    tab = bootstrap_delta_table(store, [(a, b)], invalid, N_BOOT)
                    raw = {k: metrics(za['y_true'], za['score'], za['score'] >= scores[a][1])[k]
                           - metrics(zb['y_true'], zb['score'], zb['score'] >= scores[b][1])[k] for k in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr')}
                    tab['delta_observed'] = tab['metric'].map(raw)
                    tab.insert(0, 'weighting', 'group_balanced' if gb else 'row');
                    for name, val in zip(('model_seed', 'split_seed', 'protocol', 'cohort', 'dataset'), key[::-1]):
                        tab.insert(0, name, val)
                    rows_out.append(tab)
                log(f'experiment A: {key} {a} vs {b} done')
partA = pd.concat(rows_out, ignore_index=True) if rows_out else pd.DataFrame()
partA.to_csv(os.path.join(OUT_DIR, 'reports', 'stepC_experimentA_paired_bootstrap.csv'), index=False)
pd.set_option('display.width', 220)
if len(partA):
    print(partA[partA.weighting == 'row'].round(4).to_string(index=False))

04:01:37 experiment A: ('CIC_V2', 'frozen', 'hash_group_disjoint', 'split123', 'model123') lightgbm vs mlp done
04:04:19 experiment A: ('CIC_V2', 'frozen', 'hash_group_disjoint', 'split123', 'model42') lightgbm vs mlp done
04:07:03 experiment A: ('CIC_V2', 'frozen', 'hash_group_disjoint', 'split123', 'model456') lightgbm vs mlp done
04:09:46 experiment A: ('CIC_V2', 'frozen', 'hash_group_disjoint', 'split42', 'model123') lightgbm vs mlp done
04:12:27 experiment A: ('CIC_V2', 'frozen', 'hash_group_disjoint', 'split42', 'model42') lightgbm vs mlp done
04:15:04 experiment A: ('CIC_V2', 'frozen', 'hash_group_disjoint', 'split42', 'model456') lightgbm vs mlp done
04:17:28 experiment A: ('CIC_V2', 'frozen', 'hash_group_disjoint', 'split456', 'model123') lightgbm vs mlp done
04:19:49 experiment A: ('CIC_V2', 'frozen', 'hash_group_disjoint', 'split456', 'model42') lightgbm vs mlp done
04:22:13 experiment A: ('CIC_V2', 'frozen', 'hash_group_disjoint', 'split456', 'model456') lightgbm vs mlp don

## Part 2: legacy frozen-benchmark predictions

In [5]:
legacy_rows, align_log = [], []
for ds in DATASETS_TO_RUN:
    split_path = LEGACY_SPLIT.format(ds=ds)
    if not os.path.exists(split_path):
        log(f'{ds}: {split_path} not found; skipping legacy part'); continue
    data = load_dataset(ds, DATA_DIR)
    sub = load_frozen_cohort(ds, data, SUBSET_ROOT)
    test_local = np.load(split_path)['test_idx']
    test_rows = sub[test_local]                         # subset-local index -> dataset row
    y_ref, fam_ref, key_ref = data['y'][test_rows], data['fam'][test_rows], data['keys'][test_rows]
    names = data['names']
    for ms in LEGACY_MODEL_SEEDS:
        scores = {}
        for model, pattern in LEGACY_MODELS.items():
            p = pattern.format(ds=ds, seed=ms)
            if not os.path.exists(p):
                align_log.append(dict(dataset=ds, model=model, model_seed=ms, status='file missing', path=p)); continue
            z = np.load(p, allow_pickle=True)
            ok_len = len(z['labels']) == len(y_ref)
            ok_lab = ok_len and np.array_equal(z['labels'].astype(np.int8), y_ref)
            # attack codes of the legacy files follow the subset bundle's factorization (first appearance in the
            # subset), so compare family NAMES rather than codes
            ok_fam = ok_len and np.array_equal(np.asarray(z['attack_classes'])[z['attack_codes']], names[fam_ref])
            if not (ok_lab and ok_fam):
                align_log.append(dict(dataset=ds, model=model, model_seed=ms, status=f'order mismatch (len {ok_len}, labels {ok_lab}, families {ok_fam})', path=p))
                continue
            thr = float(np.asarray(z['threshold']).ravel()[0]) if 'threshold' in z.files else None
            if thr is None:   # step-5 files store predictions but no threshold: recover it as the smallest positive score
                pos = z['scores'][z['predictions'] == 1]
                thr = float(pos.min()) if len(pos) else float('inf')
            scores[model] = (z['scores'].astype(np.float32), thr)
            align_log.append(dict(dataset=ds, model=model, model_seed=ms, status='aligned', path=p))
        for a, b in LEGACY_PAIRS:
            if a in scores and b in scores:
                for gb in ([False, True] if GROUP_BALANCED_TOO else [False]):
                    store, invalid = paired_group_bootstrap(key_ref, y_ref, {a: scores[a], b: scores[b]}, n_boot=N_BOOT, seed=BOOT_SEED, group_balanced=gb)
                    tab = bootstrap_delta_table(store, [(a, b)], invalid, N_BOOT)
                    raw = {k: metrics(y_ref, scores[a][0], scores[a][0] >= scores[a][1])[k]
                           - metrics(y_ref, scores[b][0], scores[b][0] >= scores[b][1])[k] for k in ('f1_macro', 'pr_auc', 'fpr_pct', 'tpr')}
                    tab['delta_observed'] = tab['metric'].map(raw)
                    tab.insert(0, 'weighting', 'group_balanced' if gb else 'row')
                    tab.insert(0, 'model_seed', ms); tab.insert(0, 'dataset', ds)
                    legacy_rows.append(tab)
                log(f'legacy {ds} seed {ms}: {a} vs {b} done')
    del data; gc.collect()
pd.DataFrame(align_log).to_csv(os.path.join(OUT_DIR, 'reports', 'stepC_legacy_alignment_log.csv'), index=False)
legacy = pd.concat(legacy_rows, ignore_index=True) if legacy_rows else pd.DataFrame()
legacy.to_csv(os.path.join(OUT_DIR, 'reports', 'stepC_legacy_paired_bootstrap.csv'), index=False)
print('Alignment log:'); print(pd.DataFrame(align_log).to_string(index=False))
if len(legacy):
    print('\nPaired bootstrap on the frozen benchmark (row weighting):')
    print(legacy[legacy.weighting == 'row'].round(4).to_string(index=False))
print('\nThese intervals are conditional on fixed models and thresholds; they describe test-group sampling only,'
      ' not training or batch variability, and one interval per model seed is reported, never an envelope.')

UNSW_V2: 2,390,275 rows read (11s)
05:06:21 UNSW_V2: groups 236,999 (audit 236,999) | repeated-row rate 90.08% | attack 3.98% | replaced values 0
05:06:23 UNSW_V2: frozen cohort 245,994 rows, 41,640 complete groups, attack 29.02%
05:07:48 legacy UNSW_V2 seed 42: hybrid vs lightgbm done
05:09:05 legacy UNSW_V2 seed 42: hybrid vs egraphsage done
05:10:21 legacy UNSW_V2 seed 42: hybrid vs no_graph done
05:11:37 legacy UNSW_V2 seed 42: hybrid vs mlp done
05:12:48 legacy UNSW_V2 seed 42: lightgbm vs egraphsage done
05:14:06 legacy UNSW_V2 seed 123: hybrid vs lightgbm done
05:15:23 legacy UNSW_V2 seed 123: hybrid vs egraphsage done
05:16:40 legacy UNSW_V2 seed 123: hybrid vs no_graph done
05:17:57 legacy UNSW_V2 seed 123: hybrid vs mlp done
05:19:07 legacy UNSW_V2 seed 123: lightgbm vs egraphsage done
05:20:28 legacy UNSW_V2 seed 456: hybrid vs lightgbm done
05:21:45 legacy UNSW_V2 seed 456: hybrid vs egraphsage done
05:23:00 legacy UNSW_V2 seed 456: hybrid vs no_graph done
05:24:15 legacy U

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


CIC_V2: 1,000,000 rows read (6s)

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


CIC_V2: 2,000,000 rows read (10s)

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


... (29 outputs left out)

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


CIC_V2: 18,000,000 rows read (69s)

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


CIC_V2: 18,893,708 rows read (73s)
05:26:52 CIC_V2: groups 8,981,062 (audit 8,981,062) | repeated-row rate 52.47% | attack 11.95% | replaced values 255
05:26:59 CIC_V2: frozen cohort 496,384 rows, 233,966 complete groups, attack 29.49%
05:29:37 legacy CIC_V2 seed 42: hybrid vs lightgbm done
05:32:09 legacy CIC_V2 seed 42: hybrid vs egraphsage done
05:34:40 legacy CIC_V2 seed 42: hybrid vs no_graph done
05:37:10 legacy CIC_V2 seed 42: hybrid vs mlp done
05:39:36 legacy CIC_V2 seed 42: lightgbm vs egraphsage done
05:42:10 legacy CIC_V2 seed 123: hybrid vs lightgbm done
05:44:42 legacy CIC_V2 seed 123: hybrid vs egraphsage done
05:47:11 legacy CIC_V2 seed 123: hybrid vs no_graph done
05:49:40 legacy CIC_V2 seed 123: hybrid vs mlp done
05:52:11 legacy CIC_V2 seed 123: lightgbm vs egraphsage done
05:54:46 legacy CIC_V2 seed 456: hybrid vs lightgbm done
05:57:17 legacy CIC_V2 seed 456: hybrid vs egraphsage done
05:59:53 legacy CIC_V2 seed 456: hybrid vs no_graph done
06:02:27 legacy CIC_V2 s